# Lesson 04 - Työkalun Käyttö -suunnittelumalli

Tässä oppitunnissa opit **Työkalun Käyttö** -suunnittelumallin tekoälyagentille Microsoft Agent Frameworkin (Python) avulla. Käsittelemme:

- Funktiotyökalujen määrittämisen `@tool` -koristelijalla ja tyypitetyillä parametreilla
- Työkalujen skeemojen tarjoamisen, jotta malli tietää, mitä kukin työkalu tekee
- Työkalun suorittamisen hallinnan `approval_mode` -asetuksella
- **Rakenteellisen tuloksen** palauttamisen Pydantic-mallien ja `response_format`in kautta

Tapauskuvaus on **matkavarausagentti**, joka voi etsiä kohteita, tarkistaa saatavuuden ja hakea lentotietoja.


## Asetus


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Työkalujen määrittäminen @tool-dekoraattorilla

`@tool`-dekoraattori muuttaa tavallisen Python-funktion työkaluksi, jota agentti voi kutsua.  
Keskeiset kohdat:

- **Docstring** muuttuu mallin näkemäksi työkalun kuvaukseksi.
- **Tyyppimerkinnät** (mukaan lukien `Annotated` kuvauksineen) määrittelevät työkalun skeeman.
- `approval_mode` ohjaa, pitääkö käyttäjän hyväksyä jokainen kutsu ennen kuin se suoritetaan.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Agentin luominen usealla työkalulla

Välitä kaikki kolme työkalua asiakkaalle, jotta malli voi kutsua tarpeen mukaan niitä, joita se tarvitsee vastatakseen käyttäjän kysymykseen.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Jäsennelty tulos työkalujen avulla

Asettamalla `response_format` Pydantic-malliksi, agentti pakotetaan palauttamaan hyvin tyypitetty JSON-objekti vapaamuotoisen tekstin sijaan. Tämä on hyödyllistä, kun alemman tason koodi tarvitsee käsitellä tulosta ohjelmallisesti.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Työkalun hyväksyntämallit

`approval_mode`-parametri `@tool`-merkinnässä määrittää, tarvitseeko työkalukutsut hyväksynnän ennen suoritusta:

| Tila | Käyttäytyminen |
|---|---|
| `"never_require"` | Työkalu suoritetaan automaattisesti — käyttäjän vahvistusta ei tarvita. |
| `"always_require"` | Jokainen kutsu täytyy käyttäjän hyväksyä ennen kuin se suoritetaan. |

Käytä `"always_require"`-tilaa työkaluissa, joilla on sivuvaikutuksia (esim. lennon varaaminen, luottokortin veloittaminen), jotta ihminen pysyy mukana prosessissa.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Yhteenveto

Tässä oppitunnissa opit, kuinka:

1. **Määritellä työkaluja** käyttämällä `@tool`-koristetta, jossa on tyypitetyt parametrit ja dokumentaatiorivit, jotka toimivat työkalun skeemana.
2. **Yhdistää useita työkaluja** niin, että agentti voi kutsua niitä peräkkäin vastatakseen monimutkaisiin kyselyihin.
3. **Palauttaa rakenteellista tulosta** välittämällä Pydantic-malli `response_format`-parametrina.
4. **Ohjata työkalun hyväksyntää** `approval_mode`-asetuksella, jotta ihminen pysyy mukana arkaluontoisissa toiminnoissa.

Nämä mallit muodostavat perustan luotettavien, tuotantovalmiiden agenttien rakentamiselle, jotka voivat turvallisesti olla vuorovaikutuksessa ulkoisten järjestelmien kanssa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
